# Formação dos pares por distância mínima e cointegração

Este notebook tem como objetivo formar pares de ações da B3 a partir do universo de ativos líquidos selecionado anteriormente.

A metodologia adotada combina duas etapas:

1. Pré-seleção por distância mínima entre preços normalizados, baseada no paper de Gatev, Goetzmann e Rouwenhorst.
2. Seleção final por cointegração, buscando pares com evidência estatística de relação de equilíbrio.

A formação dos pares será feita por janela temporal..

A formação não será restrita por setor. As informações setoriais serão preservadas para análise posterior.

### Importação das bibliotecas

Nesta etapa, importamos as bibliotecas necessárias para trabalhar com dados, criar combinações de pares e aplicar o teste de cointegração.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
from itertools import combinations

from statsmodels.tsa.stattools import coint
from statsmodels.api import OLS, add_constant

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.4f}".format)

### Definição dos caminhos dos arquivos

Nesta etapa, definimos os arquivos de entrada e saída.

Usaremos:

- a base de preços ajustados com informações cadastrais e setoriais;
- a base de ativos líquidos por janela.

A saída será a base com os 20 pares mais cointegrados por janela.

In [2]:
arquivo_precos = Path("../dados_tratados/dados_economatica_B3_com_setores.parquet")

arquivo_liquidez = Path("../dados_tratados/liquidez_historica.parquet")

arquivo_saida = Path("../dados_tratados/pares_top20_cointegracao.parquet")

print("Arquivo de preços existe?", arquivo_precos.exists())
print("Arquivo de liquidez existe?", arquivo_liquidez.exists())

arquivo_saida.parent.mkdir(parents=True, exist_ok=True)

Arquivo de preços existe? True
Arquivo de liquidez existe? True


### Carregamento das bases

Nesta etapa, carregamos a base de preços e a base de liquidez histórica.

A base de preços será usada para construir as séries dos ativos. A base de liquidez será usada para definir quais ativos podem entrar na formação dos pares em cada janela.

In [3]:
precos = pd.read_parquet(arquivo_precos)

liquidez = pd.read_parquet(arquivo_liquidez)

print("Base de preços:", precos.shape)
print("Base de liquidez:", liquidez.shape)

display(precos.head())
display(liquidez.head())

Base de preços: (1370766, 21)
Base de liquidez: (48752, 20)


,ticker,data,ativo,fechamento_ajustado,abertura_ajustada,minimo_ajustado,maximo_ajustado,medio_ajustado,q_negs,volume_financeiro,q_titulos,nome,classe,codigo_economatica,isin,id_papel,cnpj,situacao_cvm,setor,subsetor,segmento
0,ALLL11,2010-01-04,ALLL11<XBSP>,17.0387,16.5291,16.3192,17.0387,16.7789,"4,463.0000","39,094,265.0000","2,328,300.0000",Rumo S.A.,UNT N2,ALLL11,-,-,02387241000160,ATIVO,Bens industriais,Transporte,Transporte ferroviário
1,GVTT3,2010-01-05,GVTT3<XBSP>,55.6500,55.5400,55.5000,55.7000,55.5000,68.0000,"286,585,779.0000","5,163,600.0000",GVT Holding,ON,GVTT3,-,-,03420904000164,CANCELADA,-,-,-
2,ALLL11,2010-01-06,ALLL11<XBSP>,17.5584,18.0780,17.3985,18.5777,18.1780,"7,646.0000","78,531,111.0000","4,318,000.0000",Rumo S.A.,UNT N2,ALLL11,-,-,02387241000160,ATIVO,Bens industriais,Transporte,Transporte ferroviário
3,ALLL11,2010-01-07,ALLL11<XBSP>,17.3185,17.5883,17.0487,17.5883,17.3485,"4,952.0000","68,732,813.0000","3,958,300.0000",Rumo S.A.,UNT N2,ALLL11,-,-,02387241000160,ATIVO,Bens industriais,Transporte,Transporte ferroviário
4,AGIN3,2010-01-08,AGIN3<XBSP>,5.8100,5.6000,5.5900,5.8600,5.7400,"4,733.0000","45,060,702.0000","7,851,500.0000",Agra Incorp,ON,AGIN3,-,-,07698047000110,CANCELADA,-,-,-


,data_fim_janela,id_papel,ticker,isin,nome,classe,situacao_cvm,setor,subsetor,segmento,volume_mediano,negocios_mediano,obs_preco,cobertura,rank_volume,rank_negocios,score_liquidez,primeira_data,ultima_data,pregoes_janela
0,2012-01-31,BRVALEACNPA3,VALE5,BRVALEACNPA3,Vale,PNA,ATIVO,Materiais básicos,Mineração,Minerais metálicos,"662,657,009.5000","19,193.5000",504,1.0000,1.0000,2.0000,3.0000,2010-01-21,2012-01-31,504
1,2012-01-31,BRPETRACNPR6,PETR4,BRPETRACNPR6,Petrobras,PN,ATIVO,Petróleo gás e biocombustíveis,Petróleo gás e biocombustíveis,Exploração refino e distribuição,"500,959,699.0000","19,891.5000",504,1.0000,2.0000,1.0000,3.0000,2010-01-21,2012-01-31,504
2,2012-01-31,BROGXPACNOR3,OGXP3,BROGXPACNOR3,OGX Petroleo,ON,CANCELADA,Petróleo gás e biocombustíveis,Petróleo gás e biocombustíveis,Exploração refino e distribuição,"259,763,936.5000","12,576.5000",504,1.0000,3.0000,3.0000,6.0000,2010-01-21,2012-01-31,504
3,2012-01-31,BRITUBACNPR1,ITUB4,BRITUBACNPR1,ItauUnibanco,PN,ATIVO,Financeiro,Intermediários financeiros,Bancos,"219,055,962.5000","10,878.5000",504,1.0000,4.0000,4.0000,8.0000,2010-01-21,2012-01-31,504
4,2012-01-31,BRB3SAACNOR6,B3SA3,BRB3SAACNOR6,B3,ON,ATIVO,Financeiro,Serviços financeiros diversos,Serviços financeiros diversos,"136,037,162.0000","10,841.5000",504,1.0000,8.0000,5.0000,13.0000,2010-01-21,2012-01-31,504


### Validação das colunas necessárias

Antes de formar os pares, verificamos se as bases possuem as colunas necessárias.

Essa validação evita erros posteriores causados por colunas ausentes ou nomes diferentes do esperado.

In [4]:
colunas_precos = [
    "data",
    "id_papel",
    "ticker",
    "setor",
    "fechamento_ajustado"
]

colunas_liquidez = [
    "data_fim_janela",
    "id_papel",
    "ticker",
    "setor"
]

faltantes_precos = [
    col for col in colunas_precos
    if col not in precos.columns
]

faltantes_liquidez = [
    col for col in colunas_liquidez
    if col not in liquidez.columns
]

if faltantes_precos:
    raise ValueError(f"Colunas faltantes na base de preços: {faltantes_precos}")

if faltantes_liquidez:
    raise ValueError(f"Colunas faltantes na base de liquidez: {faltantes_liquidez}")

print("Todas as colunas necessárias estão presentes.")

Todas as colunas necessárias estão presentes.


### Preparação das bases

Garantimos que as datas estejam no formato correto, removemos observações sem preço ajustado e ordenamos as bases.

A formação dos pares será feita usando o preço de fechamento ajustado.

In [6]:
precos = precos.copy()
liquidez = liquidez.copy()

precos["data"] = pd.to_datetime(precos["data"], errors="coerce")
liquidez["data_fim_janela"] = pd.to_datetime(liquidez["data_fim_janela"], errors="coerce")

precos = precos.dropna(subset=["data", "id_papel", "fechamento_ajustado"])
liquidez = liquidez.dropna(subset=["data_fim_janela", "id_papel"])

precos = precos.sort_values(["data", "id_papel"]).reset_index(drop=True)
liquidez = liquidez.sort_values(["data_fim_janela", "id_papel"]).reset_index(drop=True)

print("Base de preços preparada:", precos.shape)
print("Base de liquidez preparada:", liquidez.shape)

Base de preços preparada: (1370766, 21)
Base de liquidez preparada: (48752, 20)


### Parâmetros da formação dos pares

Definimos os principais parâmetros da formação dos pares.

A janela de formação terá 504 pregões. Dentro de cada janela, primeiro serão selecionados candidatos por distância mínima. Depois, aplicaremos o teste de cointegração nesses candidatos.

Ao final, serão selecionados os 20 pares mais cointegrados por janela.

In [7]:
JANELA_FORMACAO = 504

COBERTURA_MINIMA_PAR = 0.80

OBS_MINIMAS_PAR = int(JANELA_FORMACAO * COBERTURA_MINIMA_PAR)

MAX_CANDIDATOS_DISTANCIA = 500

PVALOR_MAXIMO = 0.05

TOP_N_PARES = 20

### Criação da matriz de preços

Transformamos a base de preços em uma matriz.

As linhas representam datas e as colunas representam ativos identificados por "id_papel".

Essa estrutura facilita a construção das séries normalizadas e o teste dos pares.

In [8]:
matriz_precos = (
    precos
    .pivot_table(
        index="data",
        columns="id_papel",
        values="fechamento_ajustado",
        aggfunc="last"
    )
    .sort_index()
)

print("Matriz de preços:", matriz_precos.shape)

display(matriz_precos.head())

Matriz de preços: (4053, 746)


id_papel,-,BRAALRACNOR6,BRABCBACNPR4,BRABEVACNOR1,BRABRECDAM15,BRADHMACNOR9,BRAEDUACNOR9,BRAELPACNOR2,BRAERIACNOR4,BRAESBACNOR7,BRAFLTACNOR1,BRAFLUACNOR9,BRAFLUACNPA2,BRAGENBDR001,BRAGROACNOR7,BRAGXYACNOR4,BRAHEBACNOR0,BRAHEBACNPA3,BRAHEBACNPB1,BRALLDACNOR3,BRALLLACNOR6,BRALOSACNOR5,BRALPAACNOR0,BRALPAACNPR7,BRALPKACNOR9,BRALSCACNOR0,BRALUPACNOR8,BRALUPACNPR5,BRALUPCDAM15,BRAMARACNOR4,BRAMBPACNOR6,BRAMBVACNPR1,BRAMERACNOR6,BRAMILACNOR0,BRAMOBACNOR9,BRAMPIACNOR1,BRANDGACNOR9,BRANIMACNOR6,BRAPERACNOR9,BRAPTIACNPR3,BRARMLACNOR1,BRARNDACNOR6,BRARTRACNOR3,BRASAIACNOR0,BRATEDACNOR1,BRAUAUACNOR1,BRAURABDR001,BRAUREACNOR9,BRAUTMACNOR8,BRAVLLACNOR5,...,BRTTENACNOR0,BRTUPYACNOR1,BRTUPYACNPR8,BRTVITACNOR4,BRTXRXACNOR3,BRTXRXACNPR0,BRUCASACNOR1,BRUCOPACNPR5,BRUGPAACNOR8,BRUGPAACNPR5,BRUNIPACNOR7,BRUNIPACNPA0,BRUNIPACNPB8,BRUOLLACNPR5,BRUSIMACNOR3,BRUSIMACNPA6,BRUSIMACNPB4,BRVALEACNOR0,BRVALEACNPA3,BRVAMOACNOR7,BRVBBRACNOR1,BRVIGRACNOR5,BRVINEACNOR9,BRVINEACNPA2,BRVINEACNPB0,BRVITTACNOR4,BRVIVAACNOR0,BRVIVRACNOR4,BRVIVTACNOR0,BRVIVTACNPR7,BRVLIDACNOR5,BRVSTEACNOR5,BRVTRUACNOR3,BRVULCACNOR2,BRVVARACNPR8,BRVVARCDAM10,BRVVEOACNOR0,BRWDCNACNOR2,BRWEGEACNOR0,BRWESTACNOR3,BRWHRLACNOR5,BRWHRLACNPR2,BRWISAACNOR4,BRWISAACNPR1,BRWIZCACNOR5,BRWLMMACNOR6,BRWLMMACNPR3,BRWMBYACNOR2,BRYDUQACNOR3,BRZAMPACNOR5
data,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2010-01-04,17.0387,NaN,3.8282,3.1813,NaN,NaN,NaN,31.2875,NaN,NaN,NaN,NaN,NaN,2.7900,4.5320,NaN,NaN,NaN,NaN,NaN,30.9466,NaN,NaN,2.0869,NaN,NaN,NaN,NaN,NaN,38.7298,NaN,6.2279,"3,863.5224",13.8521,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.8840,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,3.4278,NaN,15.9773,NaN,3.0000,NaN,NaN,NaN,19.4598,4.2525,4.0004,1.7853,9.2953,19.6227,18.8087,NaN,18.7397,29.7739,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"10,547.9470",5.5471,18.2438,7.5823,101.5135,NaN,13.2732,NaN,NaN,NaN,NaN,1.8853,NaN,NaN,0.8348,0.7100,0.7700,NaN,NaN,3.1317,NaN,5.0762,NaN
2010-01-05,55.6500,NaN,3.9305,3.2000,NaN,NaN,NaN,32.6284,NaN,NaN,NaN,NaN,NaN,2.8300,NaN,NaN,NaN,NaN,NaN,NaN,31.1903,NaN,NaN,2.0942,NaN,NaN,NaN,NaN,NaN,40.2388,NaN,6.2019,"3,771.3162",13.6360,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.8160,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,3.4911,NaN,16.7159,NaN,3.0000,NaN,NaN,NaN,19.6747,4.2715,NaN,1.9262,9.3452,19.7458,18.8640,NaN,18.9143,30.1913,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"10,487.1519",5.5266,18.1171,7.5744,99.6622,NaN,13.4325,NaN,NaN,NaN,NaN,1.8812,NaN,NaN,0.8277,NaN,NaN,NaN,NaN,3.1713,NaN,5.1370,NaN
2010-01-06,17.5584,NaN,3.9956,3.2327,NaN,NaN,NaN,32.6582,NaN,NaN,NaN,NaN,NaN,2.8300,NaN,NaN,NaN,NaN,NaN,NaN,32.4087,NaN,NaN,2.0887,NaN,NaN,NaN,NaN,NaN,40.2052,NaN,6.2289,"3,812.8505",13.9504,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.8109,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,3.3803,NaN,16.2300,NaN,3.3000,NaN,7.5798,NaN,20.0090,4.1576,4.0004,1.8949,9.3253,19.8458,18.8456,NaN,19.3147,30.7865,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"11,155.8978",5.4326,18.2818,7.6019,99.0451,NaN,13.2732,NaN,NaN,NaN,NaN,1.8915,NaN,NaN,0.8159,NaN,NaN,NaN,NaN,3.2506,NaN,5.2209,NaN
2010-01-07,17.3185,NaN,3.9367,3.2478,NaN,NaN,NaN,32.6582,NaN,NaN,NaN,NaN,NaN,2.8000,4.6608,NaN,NaN,NaN,NaN,NaN,32.3112,NaN,2.2181,2.1105,NaN,NaN,NaN,NaN,NaN,41.1777,NaN,6.2107,"3,856.0462",14.4416,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.8126,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,3.4278,NaN,16.2300,NaN,3.2000,NaN,7.5699,NaN,19.8729,4.0247,NaN,1.8322,9.3253,19.7535,19.0226,NaN,19.3948,30.9917,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"11,247.0905",5.4326,18.1889,7.6215,98.7365,NaN,13.6449,NaN,NaN,NaN,NaN,1.9234,NaN,NaN,0.8135,NaN,NaN,NaN,NaN,3.2625,NaN,5.1370,NaN
2010-01-08,5.8100,NaN,3.9057,3.2416,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.7700,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.2573,2.1323,NaN,NaN,NaN,NaN,NaN,43.1896,NaN,6.2244,"3,763.8400",14.4907,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.9691,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,3.3777,NaN,15.9384,NaN,3.4000,NaN,7.5699,NaN,19.7941,4.2145,4.0004,1.8166,10.2438,20.0074,18.7534,NaN,19.5840,31.1628,NaN,NaN,NaN,NaN,NaN,Na

### Função para obter a janela de formação

Esta função recebe uma data final e retorna os últimos pregões disponíveis até essa data.

Ela garante que a formação dos pares use apenas informações passadas, evitando look-ahead bias.

In [9]:
def obter_janela_formacao(matriz, data_fim, tamanho_janela):
    matriz_ate_data = matriz.loc[matriz.index <= data_fim]
    
    janela = matriz_ate_data.tail(tamanho_janela)
    
    return janela

### Função para criar o índice de preços normalizados

Criamos uma função para transformar os preços ajustados em índices normalizados.

A normalização faz todos os ativos começarem em 1 dentro da janela. Assim, conseguimos comparar o comportamento relativo dos preços, independentemente do preço nominal de cada ação.

Fórmula:

preço normalizado = preço ajustado do dia / primeiro preço válido da janela

In [10]:
def criar_indice_normalizado(janela_precos):
    primeiro_preco = janela_precos.apply(
        lambda coluna: coluna.dropna().iloc[0] if coluna.dropna().shape[0] > 0 else np.nan
    )
    
    indice_normalizado = janela_precos / primeiro_preco
    
    return indice_normalizado

### Função para calcular a distância entre dois ativos

Criamos a função que calcula a distância entre dois índices normalizados.

A distância soma dos desvios quadráticos entre as duas séries normalizadas.

Quanto menor a distância, mais parecidos foram os movimentos históricos dos dois ativos.

Fórmula:
distância = soma de (índice_ativo_1 - índice_ativo_2)²

In [11]:
def calcular_distancia_par(indice_normalizado, ativo_1, ativo_2):
    dados = indice_normalizado[[ativo_1, ativo_2]].dropna()
    
    if len(dados) < OBS_MINIMAS_PAR:
        return None
    
    diferenca = dados[ativo_1] - dados[ativo_2]
    
    distancia = np.sum(diferenca ** 2)
    
    return distancia

### Função para pré-selecionar candidatos por distância mínima

Nesta etapa, criamos a função que calcula a distância entre todos os pares possíveis de uma janela e mantém apenas os candidatos mais próximos.

Essa etapa reduz o número de pares que serão submetidos ao teste de cointegração.

In [12]:
def preselecionar_candidatos_distancia(data_fim):
    janela_precos = obter_janela_formacao(
        matriz=matriz_precos,
        data_fim=data_fim,
        tamanho_janela=JANELA_FORMACAO
    )
    
    ativos_janela = (
        liquidez
        .loc[liquidez["data_fim_janela"] == data_fim, "id_papel"]
        .dropna()
        .unique()
    )
    
    ativos_janela = [
        ativo for ativo in ativos_janela
        if ativo in janela_precos.columns
    ]
    
    janela_precos = janela_precos[ativos_janela]
    
    indice_normalizado = criar_indice_normalizado(janela_precos)
    
    resultados = []
    
    for ativo_1, ativo_2 in combinations(ativos_janela, 2):
        distancia = calcular_distancia_par(
            indice_normalizado,
            ativo_1,
            ativo_2
        )
        
        if distancia is None:
            continue
        
        resultados.append({
            "data_fim_janela": data_fim,
            "ativo_1": ativo_1,
            "ativo_2": ativo_2,
            "distancia": distancia
        })
    
    candidatos = pd.DataFrame(resultados)
    
    if candidatos.empty:
        return candidatos
    
    candidatos = (
        candidatos
        .sort_values("distancia")
        .head(MAX_CANDIDATOS_DISTANCIA)
        .reset_index(drop=True)
    )
    
    candidatos["ranking_distancia"] = candidatos.index + 1
    
    return candidatos

### Função para testar cointegração de um par

Nesta etapa, criamos a função que aplica o teste de cointegração aos pares candidatos.

Para cada par, usamos o log dos preços ajustados, aplicamos o teste de cointegração e estimamos os parâmetros do spread.

O par será aprovado apenas se o p-valor da cointegração for menor ou igual a 5%.

In [13]:
def testar_cointegracao_par(janela_precos, ativo_1, ativo_2):
    dados = janela_precos[[ativo_1, ativo_2]].dropna()
    
    dados = dados[(dados[ativo_1] > 0) & (dados[ativo_2] > 0)]
    
    if len(dados) < OBS_MINIMAS_PAR:
        return None
    
    log_1 = np.log(dados[ativo_1])
    log_2 = np.log(dados[ativo_2])
    
    estatistica, pvalor, _ = coint(log_1, log_2)
    
    if pvalor > PVALOR_MAXIMO:
        return None
    
    modelo = OLS(log_1, add_constant(log_2)).fit()
    
    alpha = modelo.params["const"]
    beta = modelo.params[ativo_2]
    
    spread = log_1 - alpha - beta * log_2
    
    resultado = {
        "pvalor_coint": pvalor,
        "estatistica_coint": estatistica,
        "alpha": alpha,
        "beta": beta,
        "media_spread": spread.mean(),
        "desvio_spread": spread.std(),
        "obs_par": len(dados)
    }
    
    return resultado

### Função para formar pares cointegrados em uma janela

Combinamos o pré-filtro por distância mínima com o teste de cointegração.

Para cada janela:

1. selecionamos candidatos com menor distância;
2. aplicamos o teste de cointegração;
3. mantemos apenas pares cointegrados;
4. ordenamos os pares pelo menor p-valor;
5. selecionamos os 20 melhores pares.

In [14]:
def formar_pares_cointegrados_janela(data_fim):
    janela_precos = obter_janela_formacao(
        matriz=matriz_precos,
        data_fim=data_fim,
        tamanho_janela=JANELA_FORMACAO
    )
    
    candidatos = preselecionar_candidatos_distancia(data_fim)
    
    if candidatos.empty:
        return pd.DataFrame()
    
    resultados = []
    
    for _, candidato in candidatos.iterrows():
        resultado_coint = testar_cointegracao_par(
            janela_precos,
            candidato["ativo_1"],
            candidato["ativo_2"]
        )
        
        if resultado_coint is None:
            continue
        
        resultado = {
            "data_fim_janela": data_fim,
            "ativo_1": candidato["ativo_1"],
            "ativo_2": candidato["ativo_2"],
            "distancia": candidato["distancia"],
            "ranking_distancia": candidato["ranking_distancia"]
        }
        
        resultado.update(resultado_coint)
        
        resultados.append(resultado)
    
    pares = pd.DataFrame(resultados)
    
    if pares.empty:
        return pares
    
    pares = (
        pares
        .sort_values(
            ["pvalor_coint", "distancia"],
            ascending=[True, True]
        )
        .head(TOP_N_PARES)
        .reset_index(drop=True)
    )
    
    pares["ranking_cointegracao"] = pares.index + 1
    
    return pares

### Teste da formação em uma janela

Esse teste permite verificar se a distância mínima, a cointegração e o ranking estão funcionando corretamente.

In [15]:
data_teste = liquidez["data_fim_janela"].min()

pares_teste = formar_pares_cointegrados_janela(data_teste)

print("Data de teste:", data_teste)
print("Quantidade de pares formados:", len(pares_teste))

display(pares_teste)

Data de teste: 2012-01-31 00:00:00
Quantidade de pares formados: 20


,data_fim_janela,ativo_1,ativo_2,distancia,ranking_distancia,pvalor_coint,estatistica_coint,alpha,beta,media_spread,desvio_spread,obs_par,ranking_cointegracao
0,2012-01-31,BRTOYBACNOR4,BRTOYBACNPR1,2.0182,49,0.0000,-6.0449,0.1912,0.9229,-0.0000,0.1186,504,1
1,2012-01-31,BRBISAACNOR8,BRRSIDACNOR8,4.0676,312,0.0000,-5.5981,-3.0100,0.8609,-0.0000,0.0419,504,2
2,2012-01-31,BRBRMLACNOR9,BRHBORACNOR3,4.3212,378,0.0000,-5.2776,0.0696,0.8387,0.0000,0.0509,504,3
3,2012-01-31,BRABCBACNPR4,BRITSAACNPR7,3.9758,288,0.0001,-5.2188,0.5768,1.3438,-0.0000,0.0566,504,4
4,2012-01-31,BRFLRYACNOR5,BRWHRLACNPR2,1.7317,24,0.0001,-5.2101,1.7896,1.0638,0.0000,0.0473,466,5
5,2012-01-31,BREALTACNPR1,BRGGBRACNOR1,4.3047,371,0.0001,-5.1194,-0.6010,0.4315,0.0000,0.0480,445,6
6,2012-01-31,BRBRPRACNOR9,BRECORACNOR8,2.0851,55,0.0001,-5.0855,2.3453,0.9832,0.0000,0.0418,458,7
7,2012-01-31,BRFESAACNPR5,BRSLEDACNPR7,4.6879,478,0.0002,-4.9505,-3.8782,0.6022,0.0000,0.0675,504,8
8,2012-01-31,BRB3SAACNOR6,BROGXPACNOR3,4.2691,362,0.0004,-4.7998,-3.2862,0.5420,-0.0000,0.0489,504,9
9,2012-01-31,BRAGROACNOR7,BRSLCEACNOR2,3.6505,229,0.0004,-4.7745,1.2259,0.4487,0.0000,0.0460,437,10


### Teste em uma janela recente

Também testamos uma janela mais recente para verificar se a formação dos pares funciona em um período com maior quantidade de ativos líquidos.

In [16]:
data_teste_recente = liquidez["data_fim_janela"].max()

pares_teste_recente = formar_pares_cointegrados_janela(data_teste_recente)

print("Data de teste recente:", data_teste_recente)
print("Quantidade de pares formados:", len(pares_teste_recente))

display(pares_teste_recente)

Data de teste recente: 2026-05-31 00:00:00
Quantidade de pares formados: 20


,data_fim_janela,ativo_1,ativo_2,distancia,ranking_distancia,pvalor_coint,estatistica_coint,alpha,beta,media_spread,desvio_spread,obs_par,ranking_cointegracao
0,2026-05-31,BRTAEEACNOR9,BRTAEECDAM10,0.0331,2,0.0000,-7.8592,-1.0300,0.9786,-0.0000,0.0046,504,1
1,2026-05-31,BRCEBRACNPA8,BRCEBRACNPB6,2.0107,56,0.0000,-5.7029,-0.0441,0.9892,0.0000,0.0267,486,2
2,2026-05-31,BRALOSACNOR5,BRRIAAACNOR2,3.5807,248,0.0000,-5.6429,1.6278,0.7849,0.0000,0.0423,504,3
3,2026-05-31,BRCGRAACNOR2,BRCGRAACNPR9,0.3709,9,0.0000,-5.5892,-0.0557,1.0219,0.0000,0.0185,478,4
4,2026-05-31,BRALUPCDAM15,BRIGTICDAM16,4.2142,380,0.0000,-5.4225,1.3902,0.6451,0.0000,0.0263,504,5
5,2026-05-31,BRRNEWACNOR8,BRRNEWACNPR5,0.7851,19,0.0001,-5.2108,0.0465,0.9371,-0.0000,0.0382,504,6
6,2026-05-31,BRALUPACNPR5,BRALUPCDAM15,0.6432,16,0.0001,-5.0649,-0.5585,0.8314,0.0000,0.0112,504,7
7,2026-05-31,BRALUPACNOR8,BRENGIACNPR7,2.4620,103,0.0002,-4.9195,0.8661,0.7327,-0.0000,0.0278,504,8
8,2026-05-31,BRCTAXACNOR3,BRSOJAACNOR9,3.8546,299,0.0004,-4.7910,-3.1701,1.4551,-0.0000,0.0960,491,9
9,2026-05-31,BRALUPACNPR5,BRCSUDACNOR5,4.6738,481,0.0005,-4.7168,0.1416,0.7661,0.0000,0.0379,504,10


### Formação dos pares em todas as janelas

Nesta etapa, aplicamos a metodologia para todas as janelas temporais.

Para cada janela:

1. fazemos o pré-filtro por distância mínima;
2. aplicamos o teste de cointegração;
3. selecionamos os 20 pares mais cointegrados;
4. salvamos os resultados.

In [17]:
datas_janelas = sorted(liquidez["data_fim_janela"].dropna().unique())

resultados_pares = []

for i, data_fim in enumerate(datas_janelas):
    pares_janela = formar_pares_cointegrados_janela(data_fim)
    
    if not pares_janela.empty:
        resultados_pares.append(pares_janela)
    
    if (i + 1) % 10 == 0:
        print(f"Janelas processadas: {i + 1} de {len(datas_janelas)}")

if resultados_pares:
    pares_top20 = pd.concat(resultados_pares, ignore_index=True)
else:
    pares_top20 = pd.DataFrame()

print("Quantidade total de pares formados:", len(pares_top20))

display(pares_top20.head())

Janelas processadas: 10 de 173
Janelas processadas: 20 de 173
Janelas processadas: 30 de 173
Janelas processadas: 40 de 173
Janelas processadas: 50 de 173
Janelas processadas: 60 de 173
Janelas processadas: 70 de 173
Janelas processadas: 80 de 173
Janelas processadas: 90 de 173
Janelas processadas: 100 de 173
Janelas processadas: 110 de 173
Janelas processadas: 120 de 173
Janelas processadas: 130 de 173
Janelas processadas: 140 de 173
Janelas processadas: 150 de 173
Janelas processadas: 160 de 173
Janelas processadas: 170 de 173
Quantidade total de pares formados: 3460


,data_fim_janela,ativo_1,ativo_2,distancia,ranking_distancia,pvalor_coint,estatistica_coint,alpha,beta,media_spread,desvio_spread,obs_par,ranking_cointegracao
0,2012-01-31,BRTOYBACNOR4,BRTOYBACNPR1,2.0182,49,0.0000,-6.0449,0.1912,0.9229,-0.0000,0.1186,504,1
1,2012-01-31,BRBISAACNOR8,BRRSIDACNOR8,4.0676,312,0.0000,-5.5981,-3.0100,0.8609,-0.0000,0.0419,504,2
2,2012-01-31,BRBRMLACNOR9,BRHBORACNOR3,4.3212,378,0.0000,-5.2776,0.0696,0.8387,0.0000,0.0509,504,3
3,2012-01-31,BRABCBACNPR4,BRITSAACNPR7,3.9758,288,0.0001,-5.2188,0.5768,1.3438,-0.0000,0.0566,504,4
4,2012-01-31,BRFLRYACNOR5,BRWHRLACNPR2,1.7317,24,0.0001,-5.2101,1.7896,1.0638,0.0000,0.0473,466,5


### Adição de ticker e setor aos pares

Nesta etapa, adicionamos o ticker e o setor de cada ativo do par.

A formação dos pares usa "id_papel", mas ticker e setor facilitam a interpretação dos resultados.

In [18]:
mapa_ativos = (
    liquidez[
        ["data_fim_janela", "id_papel", "ticker", "setor"]
    ]
    .drop_duplicates()
)

pares_top20 = pares_top20.merge(
    mapa_ativos.rename(columns={
        "id_papel": "ativo_1",
        "ticker": "ticker_1",
        "setor": "setor_1"
    }),
    on=["data_fim_janela", "ativo_1"],
    how="left"
)

pares_top20 = pares_top20.merge(
    mapa_ativos.rename(columns={
        "id_papel": "ativo_2",
        "ticker": "ticker_2",
        "setor": "setor_2"
    }),
    on=["data_fim_janela", "ativo_2"],
    how="left"
)

display(pares_top20.head())

,data_fim_janela,ativo_1,ativo_2,distancia,ranking_distancia,pvalor_coint,estatistica_coint,alpha,beta,media_spread,desvio_spread,obs_par,ranking_cointegracao,ticker_1,setor_1,ticker_2,setor_2
0,2012-01-31,BRTOYBACNOR4,BRTOYBACNPR1,2.0182,49,0.0000,-6.0449,0.1912,0.9229,-0.0000,0.1186,504,1,TOYB3,Consumo cíclico,TOYB4,Consumo cíclico
1,2012-01-31,BRBISAACNOR8,BRRSIDACNOR8,4.0676,312,0.0000,-5.5981,-3.0100,0.8609,-0.0000,0.0419,504,2,BISA3,-,RSID3,Consumo cíclico
2,2012-01-31,BRBRMLACNOR9,BRHBORACNOR3,4.3212,378,0.0000,-5.2776,0.0696,0.8387,0.0000,0.0509,504,3,BRML3,Financeiro,HBOR3,Consumo cíclico
3,2012-01-31,BRABCBACNPR4,BRITSAACNPR7,3.9758,288,0.0001,-5.2188,0.5768,1.3438,-0.0000,0.0566,504,4,ABCB4,Financeiro,ITSA4,Financeiro
4,2012-01-31,BRFLRYACNOR5,BRWHRLACNPR2,1.7317,24,0.0001,-5.2101,1.7896,1.0638,0.0000,0.0473,466,5,FLRY3,Saúde,WHRL4,Consumo cíclico


### Classificação setorial dos pares

Nesta etapa, criamos colunas para indicar se os dois ativos pertencem ao mesmo setor e qual é a combinação setorial do par.

In [21]:
pares_top20["mesmo_setor"] = pares_top20["setor_1"] == pares_top20["setor_2"]

pares_top20["combinacao_setorial"] = (
    pares_top20["setor_1"].astype(str)
    + " | "
    + pares_top20["setor_2"].astype(str)
)

display(pares_top20.head())

,data_fim_janela,ranking_cointegracao,ranking_distancia,ativo_1,ticker_1,setor_1,ativo_2,ticker_2,setor_2,mesmo_setor,combinacao_setorial,distancia,pvalor_coint,estatistica_coint,alpha,beta,media_spread,desvio_spread,obs_par
0,2012-01-31,1,49,BRTOYBACNOR4,TOYB3,Consumo cíclico,BRTOYBACNPR1,TOYB4,Consumo cíclico,True,Consumo cíclico | Consumo cíclico,2.0182,0.0000,-6.0449,0.1912,0.9229,-0.0000,0.1186,504
1,2012-01-31,2,312,BRBISAACNOR8,BISA3,-,BRRSIDACNOR8,RSID3,Consumo cíclico,False,- | Consumo cíclico,4.0676,0.0000,-5.5981,-3.0100,0.8609,-0.0000,0.0419,504
2,2012-01-31,3,378,BRBRMLACNOR9,BRML3,Financeiro,BRHBORACNOR3,HBOR3,Consumo cíclico,False,Financeiro | Consumo cíclico,4.3212,0.0000,-5.2776,0.0696,0.8387,0.0000,0.0509,504
3,2012-01-31,4,288,BRABCBACNPR4,ABCB4,Financeiro,BRITSAACNPR7,ITSA4,Financeiro,True,Financeiro | Financeiro,3.9758,0.0001,-5.2188,0.5768,1.3438,-0.0000,0.0566,504
4,2012-01-31,5,24,BRFLRYACNOR5,FLRY3,Saúde,BRWHRLACNPR2,WHRL4,Consumo cíclico,False,Saúde | Consumo cíclico,1.7317,0.0001,-5.2101,1.7896,1.0638,0.0000,0.0473,466


### Organização da base final de pares

Cada linha representa um par selecionado em uma determinada janela de formação.

Como cada janela possui até 20 pares, a base principal será mantida com índice numérico simples para facilitar o salvamento e o uso nos próximos notebooks.

Além disso, criaremos uma versão apenas para visualização com índice composto por:

- data_fim_janela;
- ranking_cointegracao.

Esse índice composto é mais organizado porque identifica cada par pela janela em que ele foi formado e pela sua posição no ranking de cointegração daquela janela.

A base principal continuará com "data_fim_janela" e "ranking_cointegracao" como colunas normais.

In [23]:
colunas_finais = [
    "data_fim_janela",
    "ranking_cointegracao",
    "ranking_distancia",
    "ativo_1",
    "ticker_1",
    "setor_1",
    "ativo_2",
    "ticker_2",
    "setor_2",
    "mesmo_setor",
    "combinacao_setorial",
    "distancia",
    "pvalor_coint",
    "estatistica_coint",
    "alpha",
    "beta",
    "media_spread",
    "desvio_spread",
    "obs_par"
]

colunas_faltantes = [
    coluna for coluna in colunas_finais
    if coluna not in pares_top20.columns
]

if colunas_faltantes:
    raise ValueError(f"Colunas faltantes na base de pares: {colunas_faltantes}")

pares_top20 = pares_top20[colunas_finais].copy()

pares_top20 = pares_top20.sort_values(
    ["data_fim_janela", "ranking_cointegracao"]
).reset_index(drop=True)

pares_top20_indexado = pares_top20.set_index(
    ["data_fim_janela", "ranking_cointegracao"]
)

display(pares_top20_indexado.head(30))

ranking_distancia       ativo_1  \
data_fim_janela ranking_cointegracao                                    
2012-01-31      1                                    49  BRTOYBACNOR4   
                2                                   312  BRBISAACNOR8   
                3                                   378  BRBRMLACNOR9   
                4                                   288  BRABCBACNPR4   
                5                                    24  BRFLRYACNOR5   
                6                                   371  BREALTACNPR1   
                7                                    55  BRBRPRACNOR9   
                8                                   478  BRFESAACNPR5   
                9                                   362  BRB3SAACNOR6   
                10                                  229  BRAGROACNOR7   
                11                                  184  BRABEVACNOR1   
                12                                  209  BRCOCEACNOR0   
                13                                  418  BREQTLACNOR0   
                14                                   71  BRCOCEACNPA3   
                15                                  101  BRABCBACNPR4   
                16                                   81  BRGGBRACNOR1   
                17                                  253  BRBMEBACNPR5   
                18                                  463  BRFIEIACNOR2   
                19                                  290  BRBMTOACNPR6   
                20                                  307  BRBMTOACNPR6   
2012-02-29      1                                    19  BRTOYBACNOR4   
                2                                   310  BRBLUTACNOR8   
                3                                   242  BRBISAACNOR8   
                4                                   322  BRCOCEACNOR0   
                5                                   211  BRCOCEACNOR0   
                6                                   336  BRLAMEACNPR6   
                7                                   463  BRHOOTACNPR9   
                8                                    67  BRBRPRACNOR9   
                9                                    16  BRHOOTACNPR9   
                10                                  214  BRAMILACNOR0   

                                     ticker_1              setor_1  \
data_fim_janela ranking_cointegracao                                 
2012-01-31      1                       TOYB3      Consumo cíclico   
                2                       BISA3                    -   
                3                       BRML3           Financeiro   
                4                       ABCB4           Financeiro   
                5                       FLRY3                Saúde   
                6                       EALT4     Bens industriais   
                7                       BRPR3           Financeiro   
                8                       FESA4    Materiais básicos   
                9                       B3SA3           Financeiro   
                10                      AGRO3  Consumo não cíclico   
                11                      ABEV3  Consumo não cíclico   
                12                      COCE3    Utilidade pública   
                13                      EQTL3    Utilidade pública   
                14                      COCE5    Utilidade pública   
                15                      ABCB4           Financeiro   
                16                      GGBR3    Materiais básicos   
                17                      BMEB4           Financeiro   
                18                      FIEI3      Consumo cíclico   
                19                      BMTO4      Consumo cíclico   
                20                      BMTO4      Consumo cíclico   
2012-02-29      1                       TOYB3      Consumo cíclico   
                2                       BLUT3               Outros   
                3    

### Checagens
1) Checagem da quantidade de pares por janela
2) Checagem de pares do mesmo setor e de setores diferentes
3) Combinações setoriais mais frequentes

In [24]:
pares_por_janela = (
    pares_top20
    .groupby("data_fim_janela")
    .size()
    .reset_index(name="qtd_pares")
)

display(pares_por_janela.describe())

display(pares_por_janela.head())

display(pares_por_janela.tail())

,data_fim_janela,qtd_pares
count,173,173.0000
mean,2019-03-31 18:18:43.699422,20.0000
min,2012-01-31 00:00:00,20.0000
25%,2015-08-31 00:00:00,20.0000
50%,2019-03-31 00:00:00,20.0000
75%,2022-10-31 00:00:00,20.0000
max,2026-05-31 00:00:00,20.0000
std,NaN,0.0000


,data_fim_janela,qtd_pares
0,2012-01-31,20
1,2012-02-29,20
2,2012-03-31,20
3,2012-04-30,20
4,2012-05-31,20


,data_fim_janela,qtd_pares
168,2026-01-31,20
169,2026-02-28,20
170,2026-03-31,20
171,2026-04-30,20
172,2026-05-31,20


In [25]:
resumo_setorial = (
    pares_top20
    .groupby("mesmo_setor")
    .size()
    .reset_index(name="qtd_pares")
)

display(resumo_setorial)

,mesmo_setor,qtd_pares
0,False,2101
1,True,1359


In [26]:
combinacoes_setoriais = (
    pares_top20
    .groupby("combinacao_setorial")
    .size()
    .reset_index(name="qtd_pares")
    .sort_values("qtd_pares", ascending=False)
)

display(combinacoes_setoriais.head(20))

,combinacao_setorial,qtd_pares
105,Utilidade pública | Utilidade pública,514
52,Financeiro | Financeiro,370
27,Consumo cíclico | Consumo cíclico,192
57,Financeiro | Utilidade pública,176
50,Financeiro | Consumo cíclico,166
48,Financeiro | Bens industriais,143
100,Utilidade pública | Financeiro,139
64,Materiais básicos | Materiais básicos,122
8,Bens industriais | Bens industriais,109
25,Consumo cíclico | Bens industriais,87


### Salvamento da base final de pares formados

Essa base contém os 20 pares mais cointegrados por janela, após pré-seleção por distância mínima.

In [27]:
pares_top20.to_parquet(arquivo_saida, index=False)

print("Base de pares salva em:", arquivo_saida)

Base de pares salva em: ..\dados_tratados\pares_top20_cointegracao.parquet
